In [2]:
# Load Dependencies
import numpy as np
import pandas as pd
from itertools import product
from scipy.interpolate import interp1d
import itertools
from itertools import product
import sys
import re
import statistics
sys.path.append('/Users/meoskin/Jupyter')

BASE_THRESHOLD = 0.95
HIATUS_THRESHOLD = 0.975
CORRELATION_THRESHOLD = 0.05
MINIMUM_EVENT_QUALITY = 1
MINIMUM_JOINT_PROBABILITY = 1e-10

In [3]:
# Define Earthquake Event class [and associated helper functions]
class Earthquake:

    #Constructor: stores earthquake name, age, PDF, and CDF for each earthquake event (i.e. Bidart.Event_2)
    def __init__(self, name, age, PDF, CDF):
        self.name = name
        self.age = np.array(age)
        self.PDF = np.array(PDF)
        self.CDF = np.array(CDF)
    #Returns single sampled age based on PDF: generate random number between 0 and 1, then sample the age from the CDF that corresponds to number
    def event(self):
        r = np.random.rand()
        return self.age[np.argmin(np.abs(self.CDF - r))]
    #Returns n sampled ages: calls event() n times, then stores results in an array
    def events(self, n):
        return np.array([self.event() for _ in range(n)])
    #Resamples earthquake age to yearly resolution: calculates age range (L), then creates list of years inside age range and interpolates event PDF and CDF to those years
    def resample(self):
        L = int(self.age[-1] - self.age[0])
        new_age = np.linspace(self.age[0], self.age[-1], L + 1)

        f_age = interp1d(self.age, self.age, fill_value="extrapolate")
        f_pdf = interp1d(self.age, self.PDF, fill_value="extrapolate")
        f_cdf = interp1d(self.age, self.CDF, fill_value="extrapolate")

        self.age = f_age(new_age)
        self.PDF = f_pdf(new_age)
        self.PDF = self.PDF / np.sum(self.PDF)
        self.CDF = f_cdf(new_age)
        
    def BC(self,other):
        this_df = pd.DataFrame({'age':self.age, 'this_pdf':self.PDF})
        that_df = pd.DataFrame({'age':other.age, 'that_pdf':other.PDF})
        overlap_df = pd.merge(this_df, that_df, on='age', how='inner')
        overlap_df['root_product'] = np.sqrt(overlap_df['this_pdf'] * overlap_df['that_pdf'])
        return(sum(overlap_df['root_product']))
                                             
    
    #Calculates joint probability of multiple earthquake events: find age spacing, and make sure both events have same spacing
    def jointprobability(self, other,name='Joint'):
        this_df = pd.DataFrame({'age':self.age, 'this_pdf':self.PDF})
        that_df = pd.DataFrame({'age':other.age, 'that_pdf':other.PDF})
        overlap_df = pd.merge(this_df, that_df, on='age', how='inner')
        overlap_df['joint_pdf'] = overlap_df['this_pdf'] * overlap_df['that_pdf']
        total_joint_probability = sum(overlap_df['joint_pdf'])
        if total_joint_probability < MINIMUM_JOINT_PROBABILITY: raise RuntimeError('Sum joint probability close to zero!')
        overlap_df['joint_pdf'] = overlap_df['joint_pdf']/total_joint_probability
        overlap_df['joint_cdf'] = np.cumsum(overlap_df['joint_pdf'])
        return Earthquake(name, overlap_df['age'],overlap_df['joint_pdf'],overlap_df['joint_cdf'])

class Candidate_Event(Earthquake):
    def __init__(self, name, event_list=[]):
        self.name = name
        self.event_list = event_list
        joint = event_list[0]
        for event in event_list[1:len(event_list)]:
            joint = joint.jointprobability(event)
        self.age = list(joint.age)
        self.PDF = list(joint.PDF)
        self.CDF = list(joint.CDF)
        
        


In [4]:
# Define paleoseismic dataset object
class paleoseismic_dataset:
    # constructor stores essential site, fault, and position information
    def __init__(self,site_name,fault_name,position,slip_threshold = 0):
        self.site_name = site_name
        self.fault_name = fault_name
        self.position = position
        self.slip_threshold = 0
        self.exclude = []
        self.event_names = []

    def load_oxcal(self,filename, name_selector='Event', base_selector='Base', 
                   hiatus_selector = 'Hiatus', delim=None):
        if delim:
            df = pd.read_csv(filename, sep=delim)
        else:
            df = pd.read_csv(filename)
    
        # Keep only posterior type
        df = df[df["type"] == "posterior"]

        # Subset for events and extract
        df_events = df[df["name"].str.contains(name_selector)]

        self.events = []

        for eventname, group in df_events.groupby("name", sort=False): 
            eventname_fixed = re.sub(r"\s+", "_", eventname)
            ages = group["value"].values
            pdf = group["probability"].values
            cdf = np.cumsum(pdf)
            sumcdf = cdf[-1]
    
            self.events.append(
                Earthquake(
                    name=self.site_name + "_" + eventname_fixed,
                    age=ages,
                    PDF=pdf / sumcdf,
                    CDF=cdf / sumcdf
                )
            )

        # Extract event names and produce generic event quality tuple:
        for event in self.events:
            self.event_names.append(event.name)
        self.event_quality = dict(zip(self.event_names, [4] * len(self.events))) # set default event quality to 4

        # Subset for basal boundary
        df_base = df[df["name"].str.contains(base_selector)]
        ages = df_base["value"].values
        pdf = df_base["probability"].values
        cdf = np.cumsum(pdf)
        self.basal_boundary = ages[np.searchsorted(cdf, 1 - BASE_THRESHOLD)]

        # Subset hiatuses
        df_hiatus = df[df["name"].str.contains(hiatus_selector)]
        self.hiatuses = []

        hiatusnumber = 1
        for hiatus, group in df_hiatus.groupby("name", sort=False):
            ages = group["value"].values
            pdf = group["probability"].values
            cdf = np.cumsum(pdf)
            sumcdf = cdf[-1]

            self.hiatuses.append(
                Earthquake(
                    name=self.site_name + "_hiatus_"+str(hiatusnumber),
                    age=ages,
                    PDF=pdf / sumcdf,
                    CDF=cdf / sumcdf
                )
            )
            hiatusnumber = hiatusnumber + 1
                
    def get_marginal_pdfs(self,minimum_event_quality = 1):
        rows = []
        for i, eq in enumerate(self.events):
            if eq.name in self.exclude:
                continue
            if self.event_quality[eq.name] < minimum_event_quality:
                continue
            df = pd.DataFrame({
                "event": self.site_name + "_" + eq.name,
                "year": eq.age,
                "prob": eq.PDF,
                "CDF": eq.CDF
                
            })
            #df = df[df["year"] > 0]
            df["site_order"] = i
            rows.append(df)
        return rows

    def event_list(self):
        output_list = []
        for i, eq in enumerate(self.events):
            if eq.name in self.exclude:
                continue
            if self.event_quality[eq.name] < MINIMUM_EVENT_QUALITY:
                continue
            output_list.append(eq)
        return(output_list)
        
    
        

    

In [5]:
# Annette Paleoseismic Site

Annette_location = 20 # km
Annette = paleoseismic_dataset("Annette","San Andreas", Annette_location)
Annette.load_oxcal('paleoseismic_sites/Annette_inverted.csv')
Annette.exclude = ['Annette_Groundshaking_Event_']


In [6]:
# Frazier Mountain Paleoseismic Site
Frazier_location = 170 # km DOUBLE CHECK
Frazier = paleoseismic_dataset("Frazier","San Andreas", Frazier_location)
Frazier.load_oxcal('paleoseismic_sites/FrazierMountain_HiatusesIncluded.csv')
Frazier.exclude = ['Frazier_Event_Historic_1857']
#print(Frazier.get_event_list())
#Frazier.get_marginal_pdfs(excluded_events)
Frazier.event_quality


{'Frazier_Event_11': 4,
 'Frazier_Event_10': 4,
 'Frazier_Event_9': 4,
 'Frazier_Event_8': 4,
 'Frazier_Event_7': 4,
 'Frazier_Event_6': 4,
 'Frazier_Event_5': 4,
 'Frazier_Event_4': 4,
 'Frazier_Event_3': 4,
 'Frazier_Event_2': 4,
 'Frazier_Event_Historic_1857': 4}

In [7]:
# Pallett Creek Paleoseismic Site

Pallett_location = 272 # km
Pallett = paleoseismic_dataset("Pallett","San Andreas", Pallett_location)
Pallett.load_oxcal('paleoseismic_sites/PallettCreek_HiatusesIncluded.csv')
print(Pallett.event_names)
Pallett.exclude = ['Pallett_Event_X', 'Pallett_Event_MRE']
Pallett.event_quality.update(Pallett_Event_C = 3, Pallett_Event_V = 5)
Pallett.event_quality
#Pallett.get_marginal_pdfs(4)

['Pallett_Event_C', 'Pallett_Event_D', 'Pallett_Event_F', 'Pallett_Event_I', 'Pallett_Event_N', 'Pallett_Event_R', 'Pallett_Event_T', 'Pallett_Event_V', 'Pallett_Event_X', 'Pallett_Event_MRE']


{'Pallett_Event_C': 3,
 'Pallett_Event_D': 4,
 'Pallett_Event_F': 4,
 'Pallett_Event_I': 4,
 'Pallett_Event_N': 4,
 'Pallett_Event_R': 4,
 'Pallett_Event_T': 4,
 'Pallett_Event_V': 5,
 'Pallett_Event_X': 4,
 'Pallett_Event_MRE': 4}

In [128]:
Pallett.events[7].BC(Frazier.events[8])


0.8469602884030543

In [90]:
# Set up master event list for enumeration of candidates

included_sites = [Annette, Frazier, Pallett]
included_site_names = [site.site_name for site in included_sites]

LEFT_END = 0
LEFT_END_MINSLIP = 0
RIGHT_END = 390
RIGHT_END_MINSLIP = 0

master_event_list = [] # earthquake objects
master_site_list = [] # paleoseismic site objects in fault-position order

for site in included_sites:
    master_event_list = master_event_list + site.event_list()
    master_site_list = master_site_list + [site] * len(site.event_list())

master_event_names = [eq.name for eq in master_event_list] # corresponding list of event names
master_site_names = [site.site_name for site in master_site_list] # corresponding list of event names

#master_event_names.index(Pallett.events[7].name) Example of how to find index of an event in matrix

total_events = len(master_event_names)

# Construct correlation matrix
correlation_matrix = np.zeros((total_events,total_events))

for i in range(total_events):
    for j in range(total_events):
        correlation_matrix[i,j] = master_event_list[i].BC(master_event_list[j])

# Recursive candidate enumeration algorithm
candidate_events = []
RIGHT = 1
LEFT = 2
def make_candidate(correlation_row, new_event, direction = RIGHT, left_events = [], right_events=[], right_end = None, right_minslip = None):
    # correlation_row: one row of the correlation matrix
    # root: event at the root of this candidate tree. All events correlate to root by definition.
    # left_events: list of events in the left dirction already added to the candidate
    # right_events: list of events in the right direction already added to the candidate
    # new_event: proposed new event at the next trench site
    # direction: are we adding an event to the RIGHT or LEFT?
    
    # STEP 1: Check new event against all right and left events to make sure that this candidate is valid.
    # If not valid, then return, otherwise continue and make event and recursively call as needed.
    new_event_index = master_event_names.index(new_event.name)
    combined_events = left_events + right_events
    for event in combined_events:
        event_index = master_event_names.index(event.name)
        # Test if new event is valid correlation to each other event in the list (except root).
        # If an invalid correlation is found, we return (end the function) and don't generate a candidate 
        if correlation_matrix[new_event_index,event_index] < CORRELATION_THRESHOLD: return

    # STEP 2. Look for the next possible event (one or more) and call make_candidate for each of these. This
    # step recursively builds the candidates. To ensure against infinite recursion:
    # If the direction == RIGHT, then we look to the right and then to the left
    # If the direction == LEFT, then we only look to the left
    if direction == RIGHT:
        # find name of rightmost site name
        rightmost_site_name = master_site_names[master_event_names.index(new_event.name)]
        # find the name of the next site or end of fault
        rightmost_site_index = included_site_names.index(rightmost_site_name)
        if rightmost_site_index == len(included_site_names) - 1:
            # We found the end of the fault, use this as the end point
            right_end = RIGHT_END
            right_minslip = RIGHT_END_MINSLIP
        else:
            # We look at the next trench site on the list and call next level of recursion if we find possible correlation
            next_trench_site = included_sites[rightmost_site_index + 1]
            next_trench_site_name = next_trench_site.site_name
            possible_correlative_event_indices = [i for i, x in enumerate(master_site_names) if x == next_trench_site.site_name]
            for event_index in possible_correlative_event_indices:
                # This is the key step in the recursion. If an event at the next trench has a possible correlation,
                # we call make_candidate with that as the next_new_event and the prior new_event added to our right_events
                # If more than one possible correlative is foumd make_candidate is called for each.
                if correlation_row[event_index] >= CORRELATION_THRESHOLD:
                    next_new_event = master_event_list[event_index]
                    make_candidate(correlation_row, next_new_event, RIGHT, left_events, right_events + [new_event])
            # Now all we have left to do is get the end point information from this trench for the present candidate,
            # which will stop here.
            right_end = next_trench_site.position
            right_minslip = next_trench_site.slip_threshold

        # Final steps, looking to the right:
        right_events = right_events + [new_event] # Copy new event to right event list. List complete for this candidate.
        new_event = right_events[0] # new_event is now the root event so we can start recursion to the left
        
    # Finished looking to right, now look left
    
    leftmost_site_name = master_site_names[master_event_names.index(new_event.name)]
    # find the name of the next site or end of fault
    leftmost_site_index = included_site_names.index(leftmost_site_name)
    if leftmost_site_index == 0:
            # We found the end of the fault, use this as the other end point
        left_end = LEFT_END
        left_minslip = LEFT_END_MINSLIP
    else:
        # We look at the next trench site on the list and call next level of recursion if we find possible correlation
        next_trench_site = included_sites[leftmost_site_index - 1]
        next_trench_site_name = next_trench_site.site_name
        possible_correlative_event_indices = [i for i, x in enumerate(master_site_names) if x == next_trench_site.site_name]
        for event_index in possible_correlative_event_indices:
            # This is the key step in the recursion. If an event at the next trench has a possible correlation,
            # we call make_candidate with that as the next_new_event and the prior new_event added to our right_events
            # If more than one possible correlative is foumd make_candidate is called for each.
            if correlation_row[event_index] >= CORRELATION_THRESHOLD:
                next_new_event = master_event_list[event_index]
                make_candidate(correlation_row, next_new_event, LEFT, left_events + [new_event], right_events, right_end, right_minslip)
            # Now all we have left to do is get the end point information from this trench for the present candidate,
            # which will stop here.
        left_end = next_trench_site.position
        left_minslip = next_trench_site.slip_threshold
    
    # Final steps, looking to the left:
    left_events = left_events + [new_event] # Copy new event to left event list. List complete for this candidate.
    left_events = left_events[1:] # Remove the root event left_events (it is preserced in right_events)
    temporary_event_name = right_events[0].name # name this event after the root event. Elaborate names later....

    # Final step: create candidate and add it to the list
    candidate_events.append(Candidate_Event(temporary_event_name, left_events[::-1] + right_events))
    candidate_events[-1].right_end = right_end
    candidate_events[-1].right_minslip = right_minslip
    candidate_events[-1].left_end = left_end
    candidate_events[-1].left_minslip = left_minslip
    

    



In [88]:
candidate_events = []
start_event = 8
make_candidate(correlation_matrix[start_event,:],master_event_list[start_event])
#correlation_matrix[6,:]

In [89]:
for x in range(len(candidate_events)):
    print([event.name for event in candidate_events[x].event_list])
#indices = [i for i, x in enumerate(master_site_names) if x == 'Pallett']
#indices

['Annette_Event_6', 'Frazier_Event_8', 'Pallett_Event_N']
['Frazier_Event_8', 'Pallett_Event_N']
['Annette_Event_6', 'Frazier_Event_8', 'Pallett_Event_R']
['Frazier_Event_8', 'Pallett_Event_R']
['Annette_Event_6', 'Frazier_Event_8']
['Frazier_Event_8']


In [81]:
for x in range(len(candidate_events)):
    print(candidate_events[x].left_end)

0
20
0
20
0
20
